# Introducción a LangChain con datos académicos

Vamos a construir un asistente que consulta las **facultades y los departamentos**
de DjangoPascual. Usaremos los datos de ejemplo que ya conocemos del proyecto.

Recorrido: **datos → modelo → herramientas → agente → memoria**.

Prepara el entorno siguiendo el `README.md` de esta carpeta y selecciona su
`.venv` como kernel. La parte de datos funciona sin clave; las consultas al modelo
requieren Internet y la clave configurada en `.env`.

## 1. Los datos, antes de agregar IA

En Django, `Department` tiene una relación con `Faculty`. Aquí representamos esa
misma relación mediante el campo `facultad_id` de un archivo JSON local.

| Facultad | Departamentos | Total |
| --- | --- | --- |
| `10`: Facultad de Ingenieria | Sistemas Digitales y Eléctrica | 2 |
| `20`: Facultad de Diseño | Diseño gráfico | 1 |

La copia en `recursos/datos_academicos.json` conserva los identificadores y nombres
del fixture original. Son datos de ejemplo, no un catálogo institucional completo.
Esta práctica se ejecuta sin iniciar Django ni crear su base de datos.

In [16]:
import json
from pathlib import Path

CARPETA = Path.cwd()
if not (CARPETA / "recursos" / "datos_academicos.json").is_file():
    CARPETA = CARPETA / "clase_langchain"

ARCHIVO_DATOS = CARPETA / "recursos" / "datos_academicos.json"
if not ARCHIVO_DATOS.is_file():
    raise FileNotFoundError("Abre clase_langchain y conserva recursos/ junto al notebook.")

datos = json.loads(ARCHIVO_DATOS.read_text(encoding="utf-8"))
for facultad in datos["facultades"]:
    print(facultad["id"], "→", facultad["nombre"])

10 → Facultad de Ingenieria
20 → Facultad de Diseño


### Una función que ya podemos comprobar

La función recibe el identificador de una facultad, busca sus departamentos y
calcula el total con `len`. Si la facultad no existe, devuelve un estado explícito.
Aquí todo lo hace Python; todavía no utilizamos un modelo.

In [17]:
def consultar_departamentos(facultad_id: str) -> dict:
    """Consulta una facultad y sus departamentos en los datos de la práctica."""
    facultad_id = facultad_id.strip()
    facultad = next(
        (f for f in datos["facultades"] if f["id"] == facultad_id),
        None,
    )
    if facultad is None:
        return {"estado": "no_encontrada", "facultad_id": facultad_id}

    departamentos = [
        d for d in datos["departamentos"] if d["facultad_id"] == facultad_id
    ]
    return {
        "estado": "ok",
        "facultad_id": facultad_id,
        "facultad": facultad["nombre"],
        "total": len(departamentos),
        "departamentos": departamentos,
    }


print(json.dumps(consultar_departamentos("10"), ensure_ascii=False, indent=2))

{
  "estado": "ok",
  "facultad_id": "10",
  "facultad": "Facultad de Ingenieria",
  "total": 2,
  "departamentos": [
    {
      "id": "111",
      "nombre": "Departamento de Sistemas Digitales",
      "facultad_id": "10"
    },
    {
      "id": "112",
      "nombre": "Departamento de Eléctrica",
      "facultad_id": "10"
    }
  ]
}


In [18]:
assert consultar_departamentos("10")["total"] == 2
assert consultar_departamentos("20")["total"] == 1
assert consultar_departamentos("999")["estado"] == "no_encontrada"
print("Las consultas coinciden con los datos de ejemplo.")

Las consultas coinciden con los datos de ejemplo.


## 2. Invocar un modelo y darle instrucciones

LangChain proporciona una interfaz para trabajar con modelos y construir agentes.
En `.env`, `MODEL_PROVIDER` permite elegir **OpenAI** (`openai`) u **OpenRouter**
(`openrouter`). Completa solo la clave de la opción elegida, siguiendo el README.

Con OpenRouter usamos `openrouter/free`, que selecciona un modelo gratuito
compatible con la solicitud. Su disponibilidad y sus límites de uso pueden variar.

Un **mensaje** tiene un rol y un contenido:

- `SystemMessage`: instrucciones sobre cómo debe responder el modelo.
- `HumanMessage`: la pregunta del usuario.
- `AIMessage`: la respuesta del modelo.

La siguiente celda comprueba la clave del proveedor elegido. Nunca la imprime.
Para cambiar de proveedor, modifica `.env`, reinicia el kernel y ejecuta las celdas
desde el principio.

In [19]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage

load_dotenv(dotenv_path=CARPETA / ".env", override=True)
PROVEEDOR = os.getenv("MODEL_PROVIDER", "openai").strip().lower()

if PROVEEDOR == "openai":
    VARIABLE_CLAVE = "OPENAI_API_KEY"
    NOMBRE_MODELO = os.getenv("OPENAI_MODEL", "").strip() or "gpt-5-nano"
    URL_API = "https://api.openai.com/v1"
elif PROVEEDOR == "openrouter":
    VARIABLE_CLAVE = "OPENROUTER_API_KEY"
    NOMBRE_MODELO = os.getenv("OPENROUTER_MODEL", "").strip() or "openrouter/free"
    URL_API = "https://openrouter.ai/api/v1"
    if NOMBRE_MODELO != "openrouter/free" and not NOMBRE_MODELO.endswith(":free"):
        raise ValueError("Usa openrouter/free o un modelo terminado en :free para la opción gratuita.")
else:
    raise ValueError("MODEL_PROVIDER debe ser openai u openrouter. Revisa el archivo .env.")

clave = os.getenv(VARIABLE_CLAVE, "").strip()
if not clave or clave.startswith("tu_clave"):
    raise ValueError(f"Configura {VARIABLE_CLAVE} en clase_langchain/.env. Consulta el README.")

# Esta demostración no utiliza un servicio de trazas.
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

# Las dos APIs aceptan el formato de Chat Completions de OpenAI.
modelo = init_chat_model(
    model=NOMBRE_MODELO,
    model_provider="openai",
    api_key=clave,
    base_url=URL_API,
    use_responses_api=False,
    max_retries=1,
)
print("Proveedor seleccionado:", PROVEEDOR)
print("Modelo configurado:", NOMBRE_MODELO)

Proveedor seleccionado: openai
Modelo configurado: gpt-5-nano


Un buen prompt indica la **tarea**, el **contexto** y el **formato**. Vamos a pedir
una explicación conceptual; todavía no proporcionamos herramientas ni los
registros al modelo.

`invoke()` ejecuta la consulta y devuelve un `AIMessage`. Su propiedad `.text`
permite leer el texto de la respuesta.

In [20]:
respuesta_modelo = modelo.invoke([
    SystemMessage(content=(
        "Eres un tutor de programación. Responde en español en tres frases. "
        "Explica relaciones entre entidades sin inventar registros de una institución."
    )),
    HumanMessage(content=(
        "En nuestro proyecto Django, Department tiene una ForeignKey a Faculty. "
        "Explica esa relación con palabras sencillas."
    )),
])
print(respuesta_modelo.text)

En Django, que Department tenga una ForeignKey a Faculty significa que cada Department pertenece a una Facultad.  
Una Facultad puede tener muchos Departments, es decir, la relación es de muchos a uno.  
En código, department.faculty te da la Facultad de ese Department y, si defines related_name='departments' en la ForeignKey, faculty.departments te devuelve todos sus Departments (si no defines related_name, usarías faculty.department_set).


## 3. Convertir nuestras consultas en herramientas

El modelo no lee el JSON por sí solo. Le daremos **herramientas**: funciones Python
que el agente puede solicitar ejecutar.

`@tool` usa el nombre de la función, su documentación y los tipos de sus parámetros
para describir cómo utilizarla. Tendremos dos herramientas:

- `listar_facultades`: permite conocer los identificadores disponibles.
- `buscar_departamentos`: utiliza la función que ya comprobamos.

La ejecución directa de una herramienta con `.invoke()` no llama al modelo.

In [21]:
from langchain.tools import tool


@tool
def listar_facultades() -> list[dict]:
    """Lista los identificadores y nombres de las facultades de la práctica."""
    return datos["facultades"]


@tool
def buscar_departamentos(facultad_id: str) -> dict:
    """Consulta una facultad por su ID de texto y devuelve sus departamentos y total.

    Si no existe, devuelve estado no_encontrada. No modifica datos.
    """
    return consultar_departamentos(facultad_id)


print("Herramienta:", buscar_departamentos.name)
print("Argumentos:", buscar_departamentos.args)
print("Facultades:", listar_facultades.invoke({}))
print("Resultado:", buscar_departamentos.invoke({"facultad_id": "10"}))

Herramienta: buscar_departamentos
Argumentos: {'facultad_id': {'title': 'Facultad Id', 'type': 'string'}}
Facultades: [{'id': '10', 'nombre': 'Facultad de Ingenieria'}, {'id': '20', 'nombre': 'Facultad de Diseño'}]
Resultado: {'estado': 'ok', 'facultad_id': '10', 'facultad': 'Facultad de Ingenieria', 'total': 2, 'departamentos': [{'id': '111', 'nombre': 'Departamento de Sistemas Digitales', 'facultad_id': '10'}, {'id': '112', 'nombre': 'Departamento de Eléctrica', 'facultad_id': '10'}]}


## 4. Crear el agente

Un **agente** combina el modelo, unas instrucciones y las herramientas disponibles.
El flujo será:

**pregunta → modelo → solicitud de herramienta → función Python → resultado → respuesta**.

El modelo elige una herramienta y sus argumentos. LangChain ejecuta la función
registrada y entrega el resultado al modelo para que redacte la respuesta.

In [22]:
from langchain.agents import create_agent

INSTRUCCIONES = """
Eres un asistente de consultas académicas. Responde en español y con brevedad.
Solo tienes los datos de ejemplo de facultades y departamentos de esta práctica.
Usa las herramientas para consultar registros; no inventes nombres, IDs ni totales.
Puedes reutilizar resultados que ya estén en los mensajes de esta conversación.
Si conoces el ID, busca sus departamentos. Si solo tienes un nombre, lista las
facultades para obtener su ID. Si falta una referencia, pide una aclaración.
Si una facultad no existe en los datos, indícalo. No puedes modificar registros.
Si preguntan por docentes o programas, explica que esta práctica no tiene esos datos.
"""

agente = create_agent(
    model=modelo,
    tools=[listar_facultades, buscar_departamentos],
    system_prompt=INSTRUCCIONES,
)

PREGUNTA_INICIAL = "Consulta los departamentos de la facultad 10 e indica sus nombres y total."
respuesta_agente = agente.invoke({
    "messages": [HumanMessage(content=PREGUNTA_INICIAL)]
})
print(respuesta_agente["messages"][-1].text)

Facultad: Facultad de Ingenieria
Total de departamentos: 2
Departamentos:
- Departamento de Sistemas Digitales
- Departamento de Eléctrica


### Comprobar qué se ejecutó

El agente devuelve un **estado** que incluye `messages`. El último mensaje contiene
la respuesta final, pero también podemos revisar las acciones y los resultados:

- `AIMessage.tool_calls`: herramienta y argumentos que solicitó el modelo.
- `ToolMessage`: resultado que devolvió la función Python.

Deben aparecer los departamentos `111` y `112`, con un total de `2`.

In [23]:
from langchain.messages import AIMessage, ToolMessage

for mensaje in respuesta_agente["messages"]:
    if isinstance(mensaje, AIMessage) and mensaje.tool_calls:
        for llamada in mensaje.tool_calls:
            print("Solicitud:", llamada["name"], llamada["args"])
    elif isinstance(mensaje, ToolMessage):
        print("Resultado de Python:", mensaje.content)

Solicitud: buscar_departamentos {'facultad_id': '10'}
Resultado de Python: {"estado": "ok", "facultad_id": "10", "facultad": "Facultad de Ingenieria", "total": 2, "departamentos": [{"id": "111", "nombre": "Departamento de Sistemas Digitales", "facultad_id": "10"}, {"id": "112", "nombre": "Departamento de Eléctrica", "facultad_id": "10"}]}


## 5. Conservar una conversación

Reutilizar la variable `agente` no recupera automáticamente los mensajes de una
llamada anterior. Sin un checkpointer, esta llamada recibe únicamente la nueva
pregunta y no sabe a qué facultad se refiere.

**Predice:** ¿debería poder identificar «esa facultad» con este único mensaje?

In [24]:
PREGUNTA_SEGUIMIENTO = "¿Cómo se llama esa facultad?"
sin_historial = agente.invoke({
    "messages": [HumanMessage(content=PREGUNTA_SEGUIMIENTO)]
})
print(sin_historial["messages"][-1].text)

¿A cuál te refieres? Hay dos facultades:

- ID 10: Facultad de Ingenieria
- ID 20: Facultad de Diseño

Dime el ID (o el nombre exacto) y te digo el nombre completo o te busco sus departamentos.


Ahora añadimos `InMemorySaver`, que conserva el estado en la memoria del proceso.
El `thread_id` identifica una conversación. Sus turnos deben usar el mismo agente
y la misma configuración.

Enviamos de nuevo la consulta inicial; después enviaremos solo la pregunta nueva.

In [25]:
from langgraph.checkpoint.memory import InMemorySaver

agente_con_memoria = create_agent(
    model=modelo,
    tools=[listar_facultades, buscar_departamentos],
    system_prompt=INSTRUCCIONES,
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "consulta-academica"}}

turno_1 = agente_con_memoria.invoke({
    "messages": [HumanMessage(content=PREGUNTA_INICIAL)]
}, config=config)
print(turno_1["messages"][-1].text)

Departamentos de la Facultad 10 (Facultad de Ingenieria):
- Departamento de Sistemas Digitales
- Departamento de Eléctrica
Total de departamentos: 2


In [26]:
turno_2 = agente_con_memoria.invoke({
    "messages": [HumanMessage(content=PREGUNTA_SEGUIMIENTO)]
}, config=config)
print(turno_2["messages"][-1].text)

Se llama Facultad de Ingenieria.


Ahora puede recuperar **Facultad de Ingenieria** de la conversación. Podemos
inspeccionar los mensajes guardados sin hacer otra consulta al modelo.

Otro `thread_id` tiene su propio historial. Esta memoria no se guarda en la base
SQLite de Django: se pierde al reiniciar el kernel o crear otro checkpointer.

In [27]:
historial = agente_con_memoria.get_state(config).values.get("messages", [])
otro_config = {"configurable": {"thread_id": "otra-consulta"}}
otro_historial = agente_con_memoria.get_state(otro_config).values.get("messages", [])

print("Mensajes en nuestra conversación:", len(historial))
print("Mensajes en la otra conversación:", len(otro_historial))
assert historial
assert not otro_historial

Mensajes en nuestra conversación: 6
Mensajes en la otra conversación: 0


## 6. Una consulta que no tiene resultado

La facultad `999` no existe en nuestros datos. La herramienta debe devolver
`no_encontrada` y el asistente debe reconocerlo, sin inventar un registro.

In [28]:
consulta_inexistente = agente_con_memoria.invoke({
    "messages": [HumanMessage(content="Consulta los departamentos de la facultad 999.")]
}, config=config)
print(consulta_inexistente["messages"][-1].text)

La facultad con ID 999 no existe en los registros de esta práctica. 

Si quieres, puedo listar las facultades disponibles para verificar IDs válidos.


## Para practicar y explicar

1. Cambia la consulta anterior por «Consulta los departamentos de la facultad 20».
   Debe aparecer el departamento `211` y un total de `1`.
2. Localiza una solicitud de herramienta y su resultado. ¿Quién eligió los argumentos
   y quién calculó el total?
3. ¿Por qué el mismo `thread_id` permite entender «esa facultad»?
4. ¿Qué cambiarías en la función de consulta para obtener los datos del ORM de Django?

Ya tenemos el asistente: **modelo + instrucciones + herramientas + memoria**.
Los registros y los totales se comprueban en Python; la redacción del modelo puede variar.

- [Herramientas de LangChain](https://docs.langchain.com/oss/python/langchain/tools).
- [Memoria de conversación](https://docs.langchain.com/oss/python/langchain/short-term-memory).